In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
dataType = "RadarComposites"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
spinup_hours = "0"

RunType = ("TRACER","WET","NSSL",spinup_hours)
# RunType = ("TRACER","WET","TEMPO",spinup_hours)

# RunType = ("TRACER","DRY","NSSL",spinup_hours)
# RunType = ("TRACER","DRY","TEMPO",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset, averageType, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if averageType == 'x':
        finalDimension = len(ModelData.latitude)
    elif averageType == 'y':
        finalDimension = len(ModelData.longitude)
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzc, finalDimension)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzf, finalDimension)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output

def GetMean_x_max(variableSubset):
    variableMean = variableSubset.max(dim=("longitude"), skipna=True).data
    return variableMean
def GetMean_y_max(variableSubset):
    variableMean = variableSubset.max(dim=("latitude"), skipna=True).data
    return variableMean

# def GetMean_x_quantile(variableSubset):
#     variableMean = variableSubset.quantile(0.9,dim=("longitude"), skipna=True).data
#     return variableMean
# def GetMean_y_quantile(variableSubset):
#     variableMean = variableSubset.quantile(0.9,dim=("latitude"), skipna=True).data
#     return variableMean

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData)

In [ ]:
#TESTING PLOTS
##############
##OTHER: (Storm-Centered Composite)

In [ ]:
# #Getting Data
# t=92
# data = ModelData.GetDataTimestep_diag(t, varName = "refl10cm")
# data = data.where(data>0)

# cmap, norm, levels, ticks = RadarPlotting_Class.GetReflectivityColormap()

In [ ]:
#Max and Quantile Composites

fig, axes = plt.subplots(
    nrows=2, ncols=2, figsize=(12, 10),
    constrained_layout=True
)

# --- Compute fields ---
field_max_lat  = data.max(dim="latitude",  skipna=True)
field_max_lon  = data.max(dim="longitude", skipna=True)
field_q_lat    = data.quantile(0.9, dim="latitude",  skipna=True)
field_q_lon    = data.quantile(0.9, dim="longitude", skipna=True)

fields = [
    (field_max_lat, "Max over latitude"),
    (field_max_lon, "Max over longitude"),
    (field_q_lat,   "0.9 Quantile over latitude"),
    (field_q_lon,   "0.9 Quantile over longitude"),
]

# --- Loop and plot ---
for ax, (fld, title) in zip(axes.flat, fields):

    cf = fld.plot(
        ax=ax,
        levels=levels,
        cmap=cmap,
        norm=norm,
        extend="both",
        add_colorbar=False
    )

    ax.set_title(title)

    # Add colorbar for each subplot
    cbar = fig.colorbar(cf, ax=ax, orientation='vertical')

    # Format reflectivity bar your custom way
    RadarPlotting_Class.FormatReflectivityColorbar(
        cbar, ticks, orientation='vertical', show_labels=False
    )


In [ ]:
# # Threshold Composite

# thresholds = [0, 20, 40, 65]

# # Build threshold pairs: (0–20), (20–40), (40–65)
# threshold_pairs = [ (thresholds[i], thresholds[i+1]) for i in range(len(thresholds)-1) ]

# fig, axes = plt.subplots(
#     nrows=3, ncols=2, figsize=(12, 12),
#     constrained_layout=True
# )

# for row, (t0, t1) in enumerate(threshold_pairs):

#     # Apply threshold mask
#     masked = data.where((data >= t0)) #& (data < t1))

#     # --- Column 1: mean over latitude ---
#     ax = axes[row, 0]
#     cf = masked.mean(dim="latitude", skipna=True).plot(ax=ax,
#         levels=levels,
#         cmap=cmap,
#         norm=norm,
#         extend="both",
#         add_colorbar=False
#     )
#     ax.set_title(f"{t0}–{t1} dBZ (mean over latitude)")

#     # Add colorbar for each subplot
#     cbar = fig.colorbar(cf, ax=ax, orientation='vertical')

#     # Format reflectivity bar your custom way
#     RadarPlotting_Class.FormatReflectivityColorbar(
#         cbar, ticks, orientation='vertical', show_labels=False
#     )

#     # --- Column 2: mean over longitude ---
#     ax = axes[row, 2-1]   # same as axes[row,1]
#     cf = masked.mean(dim="longitude", skipna=True).plot(ax=ax,
#         levels=levels,
#         cmap=cmap,
#         norm=norm,
#         extend="both",
#         add_colorbar=False
#     )
#     ax.set_title(f"{t0}–{t1} dBZ (mean over longitude)")

#     # Add colorbar for each subplot
#     cbar = fig.colorbar(cf, ax=ax, orientation='vertical')

#     # Format reflectivity bar your custom way
#     RadarPlotting_Class.FormatReflectivityColorbar(
#         cbar, ticks, orientation='vertical', show_labels=False
#     )

In [ ]:
# # Threshold + Max Composite

# thresholds = [0, 20, 40, 65]

# # Build threshold pairs: (0–20), (20–40), (40–65)
# threshold_pairs = [ (thresholds[i], thresholds[i+1]) for i in range(len(thresholds)-1) ]

# fig, axes = plt.subplots(
#     nrows=3, ncols=2, figsize=(12, 12),
#     constrained_layout=True
# )

# for row, (t0, t1) in enumerate(threshold_pairs):

#     # Apply threshold mask
#     masked = data.where((data >= t0))# & (data < t1))

#     # --- Column 1: mean over latitude ---
#     ax = axes[row, 0]
#     cf = masked.max(dim="latitude", skipna=True).plot(ax=ax,
#         levels=levels,
#         cmap=cmap,
#         norm=norm,
#         extend="both",
#         add_colorbar=False
#     )
#     ax.set_title(f"{t0} dBZ (mean over latitude)")

#     # Add colorbar for each subplot
#     cbar = fig.colorbar(cf, ax=ax, orientation='vertical')

#     # Format reflectivity bar your custom way
#     RadarPlotting_Class.FormatReflectivityColorbar(
#         cbar, ticks, orientation='vertical', show_labels=False
#     )

#     # --- Column 2: mean over longitude ---
#     ax = axes[row, 2-1]   # same as axes[row,1]
#     cf = masked.max(dim="longitude", skipna=True).plot(ax=ax,
#         levels=levels,
#         cmap=cmap,
#         norm=norm,
#         extend="both",
#         add_colorbar=False
#     )
#     ax.set_title(f"{t0} dBZ (mean over longitude)")

#     # Add colorbar for each subplot
#     cbar = fig.colorbar(cf, ax=ax, orientation='vertical')

#     # Format reflectivity bar your custom way
#     RadarPlotting_Class.FormatReflectivityColorbar(
#         cbar, ticks, orientation='vertical', show_labels=False
#     )

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

def MeanDBZ(variableSubset, GetMean):
    # Convert from dBZ → linear Z (mm^6 m^-3)
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take mean in linear space
    variableMean = GetMean(variableSubset_power)

    # Convert mean Z → back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
def RunCalculations(varNames, averageType):

    if averageType == 'x':
        GetMean = GetMean_x_max
    elif averageType == 'y':
        GetMean = GetMean_y_max
    
    outputDictionary={}
    
    num_times = ModelData.Ntime
    for count, t in enumerate(tqdm(range(num_times), desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
            #Subsetting Data

            variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)

            if varName in ['refl10cm']:
                variableSubset = variableSubset.where(variableSubset > 0)

            #Applying RadarDataMask
            variableSubset = variableSubset.where(RadarDataMask == True)
            
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset, averageType, fill_nan=False)
                outputDictionary[varName] = output

            #Taking Mean
            if varName in ['refl10cm']:
                variableMean = MeanDBZ(variableSubset, GetMean)  
            else:
                variableMean = GetMean(variableSubset)
                
            outputDictionary[varName][t] = variableMean

    return outputDictionary

# Notes:
# (1) may need to subset land/water later

In [ ]:
def RunAreaAverages(ModelData,varNames,averageType,name):
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName = f"outputDictionary_{name}.h5")
    
    #loading back in 
    try:
        outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
        return outputDictionary
    except Exception as e:
        print(f"Error: {e}")
        
        print("Running Calculation")
        outputDictionary = RunCalculations(varNames,averageType) #takes about 10 minutes
        #saving output
        
        DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)
        return outputDictionary

In [ ]:
###############
#Loading in MRMS RadarTimeseries
###############

def LoadRadarTimeseries(ModelData):
    """
    Build the time-series filename using ModelData and load the .pkl file.
    Creates output directory if needed.
    """

    # Build file name
    fileName = (
        f"RadarTimeseries_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs.pkl"
    )

    # Build directory for radar timeseries
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType='DataAnalysis/Observation_Data', dataType='RadarComparison'),
        "RadarTimeseries"
    )
    os.makedirs(outputDir, exist_ok=True)

    # Full path to the .pkl file
    fullFilePath = os.path.join(outputDir, fileName)

    # Try to load existing file
    if os.path.exists(fullFilePath):
        print(f"Loading existing file: {fullFilePath}")
        with open(fullFilePath, "rb") as f:
            return fullFilePath, pickle.load(f)

    # No file found
    return fullFilePath, None

def Add_MRMS_RadarTimeSeries_Plot(ax, loc='lower right'):
    ax.plot([datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings], MRMS_RadarTimeseries, color='black',label='MRMS')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, frameon=True, fontsize=9, loc=loc)

fileName, list_array = LoadRadarTimeseries(ModelData)
if list_array is not None:
    MRMS_RadarTimeseries = list_array[:,2]

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
def GetVarNames():
    #3D Variables (9 vars)
    #microphysics variables
    varNames = ["refl10cm"]
    return varNames

#running
def GetDictionary_x(ModelData):
    varNames = GetVarNames()
    outputDictionary = RunAreaAverages(ModelData,varNames, "x", "1")
    return outputDictionary

#running
def GetDictionary_y(ModelData):
    varNames = GetVarNames()
    outputDictionary = RunAreaAverages(ModelData,varNames, "y", "2")
    return outputDictionary

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
def MakePlots(ModelData,
              outputDictionary_y_NSSL,
              outputDictionary_y_TEMPO,
              outputDictionary_x_NSSL,
              outputDictionary_x_TEMPO,
              levels, cmap, norm, ticks,
              useTime=True,          # True = plot at specific time
              t=60,                   # which time index to use
              figsize=(12,10)):

    # ---- Coordinates ----
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3
    zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
    lat = ModelData.latitude
    lon = ModelData.longitude

    # ---- Extract reflectivity arrays ----
    varList = [
        outputDictionary_y_NSSL['refl10cm'],
        outputDictionary_y_TEMPO['refl10cm'],
        outputDictionary_x_NSSL['refl10cm'],
        outputDictionary_x_TEMPO['refl10cm']
    ]

    titles = ["NSSL","TEMPO","NSSL","TEMPO"]

    # ---- Create 2x2 grid ----
    fig, axes = plt.subplots(
        nrows=2, ncols=2,
        figsize=figsize,
        constrained_layout=True
    )

    # ---- Select time or time-mean ----
    if useTime:
        varTimeList = [v[t] for v in varList]
        suptitleText = f"Max Reflectivity Composite at {ModelData.timeStrings[t]}"
    else:
        varTimeList = [np.nanmean(v, axis=0) for v in varList]
        suptitleText = "Max Reflectivity Composite Time-Average"

    # ---- Loop through the panels ----
    for idx, ax in enumerate(axes.flatten()):
        varTimemean = varTimeList[idx]

        # Coordinate to plot along x-axis
        if idx < 2:     # y-avg → x-axis is longitude
            xAxis = lon
        else:           # x-avg → x-axis is latitude
            xAxis = lat

        # y-axis index
        # yAxis = np.arange(varTimemean.shape[0])
        yAxis = zlevels_center

        # ---- Contourf ----
        cf = ax.contourf(
            xAxis,
            yAxis,
            varTimemean,
            levels=levels,
            cmap=cmap,
            norm=norm,
            extend="both"
        )

        cbar = fig.colorbar(cf, ax=ax, orientation='vertical')
        RadarPlotting_Class.FormatReflectivityColorbar(
            cbar, ticks, orientation='vertical', show_labels=False
        )

        ax.set_title(titles[idx])
        ax.set_xlabel("Longitude" if idx < 2 else "Latitude")
        ax.set_ylabel("Altitude (km)")
        ax.set_ylim(top=20)

    # ---- Final Title ----
    fig.suptitle(suptitleText)

    return fig, axes


In [ ]:
def SaveFigure(fig,label):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFile = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RadarComposites_NSSLvsTEMPO_{label}.jpg"
    )

    # --- Save figure ---
    fig.savefig(outputFile, dpi=100, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
####################################
#CALCULATING

In [ ]:
#getting NSSL dictionaries
RunType = ("TRACER","WET","NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

outputDictionary_x_NSSL = GetDictionary_x(ModelData)
outputDictionary_y_NSSL = GetDictionary_y(ModelData)


#getting TEMPO dictionaries
RunType = ("TRACER","WET","TEMPO",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

outputDictionary_x_TEMPO = GetDictionary_x(ModelData)
outputDictionary_y_TEMPO = GetDictionary_y(ModelData)

In [ ]:
####################################
#PLOTTING

In [ ]:
t=np.where(np.array(ModelData.timeStrings) == '2022-07-01_15.00.00')[0][0]
fig, axes = MakePlots(
    ModelData,
    outputDictionary_y_NSSL,
    outputDictionary_y_TEMPO,
    outputDictionary_x_NSSL,
    outputDictionary_x_TEMPO,
    levels, cmap, norm, ticks,
    useTime=True,t=t)
SaveFigure(fig,label=ModelData.timeStrings[t])

In [ ]:
fig, axes = MakePlots(
    ModelData,
    outputDictionary_y_NSSL,
    outputDictionary_y_TEMPO,
    outputDictionary_x_NSSL,
    outputDictionary_x_TEMPO,
    levels, cmap, norm, ticks,
    useTime=False)
SaveFigure(fig,label="Time-Average")